# 02 · Train and evaluate an image model

This notebook trains a model on the split saved by notebook 01,
following the Week 9, 11 and 12 slides. Choose the model with `MODEL`:

| `MODEL` | What it is | Course week |
|---|---|---|
| `"resnet18"` | Pretrained ResNet-18: transfer learning. The default | 12 |
| `"small_cnn"` | The three-block CNN, trained from scratch | 11 |
| `"mlp"` | The image flattened into one row: your **MLP baseline** | 11 |

The final project asks for two baselines: always answering the largest
class (Step 5 does it every run) and an MLP (one run with
`MODEL = "mlp"`). Compare both with your CNN in `experiments.csv`.

The steps:

| Step | What happens | Course week |
|---|---|---|
| 1–4 | Setup, settings, data loaders | 12 |
| 5 | Baseline: always answer the largest class | 10 |
| 6 | Check: can the model overfit one batch? | 9 |
| 7 | Train. ResNet-18, phase 1: only the new head. Other models: everything | 11, 12 |
| 8 | ResNet-18, phase 2: fine-tune the whole network | 12 |
| 9 | Learning curves | 9 |
| 10 | Write the run in the experiment log | 9 |
| 11 | Test set, **once**: accuracy, confusion matrix, worst mistakes | 10 |
| 12 | Save the files the Streamlit app needs | 13 |

**Before you start:** run notebook 01 once. In Colab, choose
**Runtime → Change runtime type → T4 GPU**: training on the CPU is
many times slower.

**Each run:** change one setting in Step 2, describe the change in
`RUN_NOTE`, run all cells, and compare the new row in
`outputs/experiments.csv` with the earlier ones.

## Step 1 · Setup
Same as notebook 01. Use the same `REPO_URL`.

In [ ]:
# Run this cell first, every time you open the notebook.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# CHANGE: the address of YOUR team's copy of this repo on GitHub
REPO_URL = "https://github.com/dsc-scsu/csc451-project-starter.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content") / Path(REPO_URL).stem
    if not REPO_DIR.exists():
        !git clone -q {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    !git pull -q
elif Path.cwd().name == "notebooks":
    os.chdir("..")  # on your own computer: work from the repo folder

sys.path.insert(0, "src")
print("Working in", Path.cwd())

## Step 2 · Settings (CHANGE)

| Setting | What it does |
|---|---|
| `PROJECT_FOLDER`, `DATA_ZIP_NAME` | Same as in notebook 01 |
| `MODEL` | `"resnet18"`, `"small_cnn"` or `"mlp"` (see the table at the top) |
| `IMAGE_SIZE` | Images are resized to this square size. Use 224 for `resnet18`, the size it was trained on. Use 64 (or 96) for `small_cnn` and `mlp`: larger images make their first Linear layer huge |
| `FLIP` | `False` if a mirror image changes the class (text, X-rays, left vs. right) |
| `BATCH_SIZE` | Images per step. Lower it if Colab runs out of memory |
| `LR_HEAD`, `EPOCHS_HEAD` | `resnet18` phase 1: learning rate and maximum epochs for the new head |
| `FINE_TUNE`, `LR_FINE`, `EPOCHS_FINE` | `resnet18` phase 2: unfreeze everything and train at a smaller rate |
| `LR_SCRATCH`, `EPOCHS_SCRATCH` | `small_cnn` and `mlp`: learning rate and maximum epochs. A model that starts from random weights needs more epochs |
| `PATIENCE` | Stop a phase after this many epochs with no gain in validation accuracy |
| `RUN_NOTE` | **Change every run.** One line saying what is different this time |
| `EVALUATE_TEST` | Leave `False` while you compare settings. Set `True` once, for your final model |

In [ ]:
# CHANGE: settings for this run
PROJECT_FOLDER = "csc451_project"
DATA_ZIP_NAME = "data.zip"

MODEL = "resnet18"   # "resnet18", "small_cnn" or "mlp"
IMAGE_SIZE = 224     # 224 for resnet18; 64 for small_cnn and mlp
FLIP = True
BATCH_SIZE = 32
PRETRAINED = True

LR_HEAD = 1e-3       # phase 1: only the new head trains
EPOCHS_HEAD = 5

FINE_TUNE = True     # phase 2: everything trains, smaller rate
LR_FINE = 1e-4
EPOCHS_FINE = 10

LR_SCRATCH = 1e-3    # small_cnn and mlp: one phase, everything trains
EPOCHS_SCRATCH = 30

PATIENCE = 4
SEED = 0

RUN_NOTE = "first run: default settings"   # CHANGE every run
EVALUATE_TEST = False                      # True only at the very end

In [ ]:
# Where things are. You should not need to change this cell.
if IN_COLAB:
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
    DATA_DIR = Path("/content/data")      # fast disk, erased after
else:                                     # the session ends
    PROJECT = Path("project_files")       # on your own computer
    DATA_DIR = PROJECT / "unzipped"
OUT = PROJECT / "outputs"                 # results to keep, on Drive
OUT.mkdir(parents=True, exist_ok=True)
print("Project folder:", PROJECT)

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch import nn
from torch.utils.data import DataLoader

import data_utils as du
import train_utils as tu

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("No GPU. In Colab: Runtime > Change runtime type > T4 GPU")

## Step 3 · Load the saved split
The data is unzipped again if this is a new Colab session. The split and the class list come from notebook 01, so this run uses exactly the same train, validation and test images as every other run.

In [ ]:
du.unzip_once(PROJECT / DATA_ZIP_NAME, DATA_DIR)
with open(OUT / "data_info.json") as f:
    IMAGE_ROOT = DATA_DIR / json.load(f)["image_root"]
with open(OUT / "classes.json") as f:
    classes = json.load(f)

table = pd.read_csv(OUT / "splits.csv")
train_table = table[table["split"] == "train"]
val_table = table[table["split"] == "val"]
test_table = table[table["split"] == "test"]
print(len(train_table), "train,", len(val_table), "val,",
      len(test_table), "test images")
print(len(classes), "classes:", classes)

## Step 4 · Datasets and loaders
Training images get random augmentation; validation and test images get the same fixed resize every time (Week 12).

In [ ]:
tu.set_seed(SEED)
train_tfms, eval_tfms = tu.make_transforms(IMAGE_SIZE, flip=FLIP)

train_ds = du.ImageDataset(train_table, IMAGE_ROOT, classes, train_tfms)
val_ds = du.ImageDataset(val_table, IMAGE_ROOT, classes, eval_tfms)
test_ds = du.ImageDataset(test_table, IMAGE_ROOT, classes, eval_tfms)

workers = 2 if IN_COLAB else 0
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=workers)
val_dl = DataLoader(val_ds, batch_size=64, num_workers=workers)
test_dl = DataLoader(test_ds, batch_size=64, num_workers=workers)

## Step 5 · Baseline
A model that ignores the image and always answers the most common training class. Your model is only useful if it clearly beats this number.

In [ ]:
largest = train_table["label"].value_counts().idxmax()
baseline_val = (val_table["label"] == largest).mean()
print(f"Always answering '{largest}': {baseline_val:.1%} "
      "validation accuracy")

## Step 6 · Check: can the model overfit one batch?
Train on a single batch for 30 steps. The loss should fall close to 0. If it does not, there is a bug in the data or the model, and more epochs will not fix it (Week 9).

In [ ]:
tu.set_seed(SEED)
check_model = tu.make_model(MODEL, len(classes), IMAGE_SIZE,
                            PRETRAINED).to(device)
check_opt = torch.optim.Adam(check_model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

xb, yb = next(iter(train_dl))
xb, yb = xb[:16].to(device), yb[:16].to(device)
check_model.train()
for step in range(30):
    check_opt.zero_grad()
    loss = loss_fn(check_model(xb), yb)
    loss.backward()
    check_opt.step()
print(f"Loss after 30 steps on one batch: {loss.item():.3f}")
print("Good: well below 0.5" if loss.item() < 0.5 else
      "Look for a bug: wrong labels, or images and labels out of order")
del check_model, check_opt

## Step 7 · Train
**`resnet18`, phase 1:** the pretrained backbone is frozen; only the new last layer learns.

**`small_cnn` and `mlp`:** every layer starts from random weights and trains in one phase.

The weights with the best validation accuracy are saved to `outputs/best.pt` after every improvement. The cell also prints the number of parameters (Week 11).

In [ ]:
tu.set_seed(SEED)
model = tu.make_model(MODEL, len(classes), IMAGE_SIZE,
                      PRETRAINED).to(device)
print(f"{MODEL}: {tu.count_parameters(model):,} parameters")
checkpoint = OUT / "best.pt"
with open(OUT / "model_info.json", "w") as f:  # the app reads this
    json.dump({"model": MODEL, "image_size": IMAGE_SIZE}, f)

if MODEL == "resnet18":
    tu.freeze_backbone(model)
    opt = torch.optim.Adam(model.fc.parameters(), lr=LR_HEAD)
    history = tu.fit(model, train_dl, val_dl, loss_fn, opt, device,
                     EPOCHS_HEAD, checkpoint, PATIENCE,
                     frozen_backbone=True)
else:
    opt = torch.optim.Adam(model.parameters(), lr=LR_SCRATCH)
    history = tu.fit(model, train_dl, val_dl, loss_fn, opt, device,
                     EPOCHS_SCRATCH, checkpoint, PATIENCE)

## Step 8 · ResNet-18, phase 2: fine-tune the whole network
Now every layer trains, at a learning rate about 10 times smaller, so the pretrained weights only move a little. Set `FINE_TUNE = False` to skip this phase. For `small_cnn` and `mlp` this phase is skipped: they already trained every layer in Step 7.

In [ ]:
if MODEL == "resnet18" and FINE_TUNE:
    tu.unfreeze_all(model)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_FINE)
    best_phase1 = max(h["val_acc"] for h in history)
    history += tu.fit(model, train_dl, val_dl, loss_fn, opt, device,
                      EPOCHS_FINE, checkpoint, PATIENCE,
                      best_acc=best_phase1)

hist = pd.DataFrame(history)
hist["epoch"] = range(1, len(hist) + 1)
best_val = hist["val_acc"].max()
print(f"Best validation accuracy: {best_val:.1%} "
      f"(baseline {baseline_val:.1%})")

## Step 9 · Learning curves
Training loss falling while validation loss rises means overfitting; both staying high means underfitting (Week 9). The plot is saved to `outputs/` so you can put it in your presentation.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist["epoch"], hist["train_loss"], label="train")
ax1.plot(hist["epoch"], hist["val_loss"], label="validation")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax1.legend()
ax2.plot(hist["epoch"], hist["train_acc"], label="train")
ax2.plot(hist["epoch"], hist["val_acc"], label="validation")
ax2.axhline(baseline_val, color="gray", linestyle="--", label="baseline")
ax2.set_xlabel("epoch")
ax2.set_ylabel("accuracy")
ax2.legend()
if MODEL == "resnet18" and FINE_TUNE:
    for ax in (ax1, ax2):
        ax.axvline(EPOCHS_HEAD + 0.5, color="lightgray")
plt.tight_layout()
plt.savefig(OUT / "learning_curves.png", dpi=150)
plt.show()

## Step 10 · Write this run in the experiment log
One row per run in `outputs/experiments.csv`. Change one thing per run and say what in `RUN_NOTE`. This log is part of the evidence the final project asks for.

In [ ]:
tu.log_run(OUT / "experiments.csv",
           note=RUN_NOTE, model=MODEL,
           image_size=IMAGE_SIZE, flip=FLIP, batch_size=BATCH_SIZE,
           lr_head=LR_HEAD, fine_tune=FINE_TUNE, lr_fine=LR_FINE,
           lr_scratch=LR_SCRATCH,
           epochs_run=len(hist), seed=SEED,
           baseline_val=round(baseline_val, 4),
           best_val_acc=round(best_val, 4))
pd.read_csv(OUT / "experiments.csv")

## Step 11 · Test set (once, at the end)

Run this only for your **final** model, after you have stopped
changing settings. If you look at the test score and then change
settings to raise it, the test set has become a second validation
set, and the score you report is too high.

Set `EVALUATE_TEST = True` in Step 2 and run all cells.

In [ ]:
if not EVALUATE_TEST:
    print("Skipped. Set EVALUATE_TEST = True for your final model.")
else:
    y_true, y_pred, probs = tu.predict_all(model, test_dl, device)
    acc = (y_true == y_pred).mean()
    baseline_test = (test_table["label"] == largest).mean()
    print(f"Test accuracy: {acc:.1%}   (baseline {baseline_test:.1%})")
    print()
    for i, c in enumerate(classes):
        mask = y_true == i
        if mask.sum() == 0:
            print(f"  {c:20s} no test images")
            continue
        print(f"  {c:20s} {(y_pred[mask] == i).mean():.1%}"
              f"  of {mask.sum()} images")

    cm = confusion_matrix(y_true, y_pred, labels=range(len(classes)))
    ConfusionMatrixDisplay(cm, display_labels=classes).plot(
        xticks_rotation=45, colorbar=False)
    plt.title("Test set: rows are the true class")
    plt.tight_layout()
    plt.savefig(OUT / "confusion_matrix.png", dpi=150)
    plt.show()

The most confidently wrong test images. Some may be mislabeled; others show what the model has not learned. Both belong in your error analysis.

In [ ]:
if EVALUATE_TEST:
    wrong = np.where(y_pred != y_true)[0]
    worst = wrong[np.argsort(-probs[wrong].max(axis=1))][:8]
    if len(worst) == 0:
        print("No mistakes on the test set.")
    else:
        fig, axes = plt.subplots(1, len(worst),
                                 figsize=(2.6 * len(worst), 3.2),
                                 squeeze=False)
        for ax, i in zip(axes[0], worst):
            img = Image.open(IMAGE_ROOT / test_table["path"].iloc[i])
            ax.imshow(img.convert("RGB"))
            ax.set_title(f"true: {classes[y_true[i]]}\n"
                         f"said: {classes[y_pred[i]]} "
                         f"{probs[i].max():.0%}", fontsize=9)
            ax.axis("off")
        plt.tight_layout()
        plt.show()

## Step 12 · Files for the Streamlit app

The app needs three files from `outputs/`:

- `best.pt`: the weights
- `classes.json`: the class names, in index order
- `model_info.json`: which model this is and its image size, so the
  app builds the same model and resizes images the same way

Copy them into the repo's `app/` folder on your laptop. In Colab,
this cell downloads them. `best.pt` is about 45 MB for ResNet-18 and
smaller for the other models, so it fits on GitHub.

In [ ]:
print("best.pt:", round(checkpoint.stat().st_size / 1e6, 1), "MB")
app_files = ["best.pt", "classes.json", "model_info.json"]
if IN_COLAB:
    from google.colab import files
    for name in app_files:
        files.download(str(OUT / name))
else:
    print("Copy these into app/:")
    for name in app_files:
        print("  ", OUT / name)